# Main Prediction

This file contains hyperparameter tuning and performance testing for four models: NB, DT, DNN(MLP), GBDT

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

## Hyperparameter tuning

In [ ]:
# Load data
train_raw_datas = pd.read_csv('./data/recipe_train.csv')
train_labels = train_raw_datas["duration_label"]
train_n_steps = train_raw_datas["n_steps"]
train_n_ingrs = train_raw_datas["n_ingredients"]

train_names_cv = sps.load_npz('./data/countvec/train_name_vec.npz')
train_steps_cv = sps.load_npz('./data/countvec/train_steps_vec.npz')
train_ingrs_cv = sps.load_npz('./data/countvec/train_ingr_vec.npz')

# Combine features into a single matrix
X = sps.hstack((
    sps.csr_matrix(train_n_steps).T,
    sps.csr_matrix(train_n_ingrs).T,
    train_names_cv,
    train_steps_cv,
    train_ingrs_cv
), format="csr")

y = train_labels

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# Function to train and evaluate models
def evaluate_model(model, param_grid=None):
    grid_search = None
    if param_grid:
        grid_search = GridSearchCV(model, param_grid, cv=2, n_jobs=-1, verbose=1, scoring='accuracy')
        grid_search.fit(X_train, y_train)
        best_params = grid_search.best_params_
        print(f"Best Parameters: {best_params}")
        model = grid_search.best_estimator_

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Training accuracy for {model.__class__.__name__}: ", model.score(X_train, y_train))
    print(f"Test accuracy for {model.__class__.__name__}: ", model.score(X_test, y_test))
    print(f"Classification Report for {model.__class__.__name__}:")
    print(classification_report(y_pred, y_test))

    return model, grid_search


In [ ]:
# Model 1: Naive Bayes (BernoulliNB) with default parameters
print("\nEvaluating Bernoulli Naive Bayes:")
model_nb, _ = evaluate_model(BernoulliNB())

In [ ]:
# Model 2: Decision Tree with hyperparameter tuning
print("\nTuning DecisionTreeClassifier:")
param_grid_tree = {'max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]}
model_tree, grid_search_tree = evaluate_model(DecisionTreeClassifier(random_state=123), param_grid_tree)

In [ ]:
# Model 2: GBDT with hyperparameter tuning
# This part need 20 minutes
print("\nTuning GradientBoostingClassifier:")
param_grid_gb = {'n_estimators': [50, 80, 100]}
model_gb, grid_search_gb = evaluate_model(GradientBoostingClassifier(random_state=123, max_depth=3), param_grid_gb)

In [ ]:
# Model 4: MLP with hyperparameter tuning
# This part need 60 minutes
print("\nTuning MLPClassifier:")
param_grid_mlp = {'hidden_layer_sizes': [(11, )]}
model_mlp, grid_search_mlp = evaluate_model(MLPClassifier(
    max_iter=10000, activation='relu', solver='sgd', alpha=0.01, random_state=123), param_grid_mlp)

In [ ]:
# Visualization for best hyperparameters in each model
def plot_grid_search_results(grid_search, title):
    results = pd.DataFrame(grid_search.cv_results_)

    results = results.sort_values('param_max_depth')
    
    plt.figure(figsize=(8, 6))
    plt.title(title)
    plt.plot(results['param_max_depth'], results['mean_test_score'], marker='o', linestyle='-', color='blue', label="mean_test_score")
    plt.xlabel('Max Depth')  
    plt.ylabel('Mean Test Score')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot Grid Search results for models
plot_grid_search_results(grid_search_tree, "DecisionTreeClassifier Hyperparameters")

## Model prediction

In [ ]:
# ********Part 1 Raw data processing********
train_raw_data = pd.read_csv('./data/recipe_train.csv')
test_raw_data = pd.read_csv('./data/recipe_test.csv')

train_labels = train_raw_data["duration_label"]
test_labels = test_raw_data["duration_label"]

In [ ]:
# ********Part 1: Load Data********
# Load the training data
train_raw_data = pd.read_csv('./data/recipe_train.csv')

# Load the test data
test_raw_data = pd.read_csv('./data/recipe_test.csv')

# Extract features from the training data
train_n_steps = train_raw_data["n_steps"]
train_n_ingrs = train_raw_data["n_ingredients"]

# CountVectorizer features for training
train_names_cv = sps.load_npz('./data/countvec/train_name_vec.npz')
train_steps_cv = sps.load_npz('./data/countvec/train_steps_vec.npz')
train_ingrs_cv = sps.load_npz('./data/countvec/train_ingr_vec.npz')

# Doc2Vec features for training
train_names_d2v50 = pd.read_csv('./data/doc2vec50/train_name_doc2vec50.csv', header=None, engine='python')
train_steps_d2v50 = pd.read_csv('./data/doc2vec50/train_steps_doc2vec50.csv', header=None, engine='python')
train_ingrs_d2v50 = pd.read_csv('./data/doc2vec50/train_ingr_doc2vec50.csv', header=None, engine='python')

train_names_d2v100 = pd.read_csv('./data/doc2vec100/train_name_doc2vec100.csv', header=None, engine='python')
train_steps_d2v100 = pd.read_csv('./data/doc2vec100/train_steps_doc2vec100.csv', header=None, engine='python')
train_ingrs_d2v100 = pd.read_csv('./data/doc2vec100/train_steps_doc2vec100.csv', header=None, engine='python')

# Extract features from the test data
test_n_steps = test_raw_data["n_steps"]
test_n_ingrs = test_raw_data["n_ingredients"]

# CountVectorizer features for test
test_names_cv = sps.load_npz('./data/countvec/test_name_vec.npz')
test_steps_cv = sps.load_npz('./data/countvec/test_steps_vec.npz')
test_ingrs_cv = sps.load_npz('./data/countvec/test_ingr_vec.npz')

# Doc2Vec features for test
test_names_d2v50 = pd.read_csv('./data/doc2vec50/test_name_doc2vec50.csv', header=None, engine='python')
test_steps_d2v50 = pd.read_csv('./data/doc2vec50/test_steps_doc2vec50.csv', header=None, engine='python')
test_ingrs_d2v50 = pd.read_csv('./data/doc2vec50/test_ingr_doc2vec50.csv', header=None, engine='python')

test_names_d2v100 = pd.read_csv('./data/doc2vec100/test_name_doc2vec100.csv', header=None, engine='python')
test_steps_d2v100 = pd.read_csv('./data/doc2vec100/test_steps_doc2vec100.csv', header=None, engine='python')
test_ingrs_d2v100 = pd.read_csv('./data/doc2vec100/test_steps_doc2vec100.csv', header=None, engine='python')

# ********Part 2: Combine features********
# Stack the train set and the test set for CountVectorizer (CSR matrix format)
X_train_cs = sps.hstack((
    sps.csr_matrix(train_n_steps).T,
    sps.csr_matrix(train_n_ingrs).T,
    train_names_cv,
    train_steps_cv,
    train_ingrs_cv
), format="csr")

X_test_cs = sps.hstack((
    sps.csr_matrix(test_n_steps).T,
    sps.csr_matrix(test_n_ingrs).T,
    test_names_cv,
    test_steps_cv,
    test_ingrs_cv
), format="csr")

# Combine features for Doc2Vec (Pandas DataFrame format)
X_train_d2v50 = pd.concat(
    [train_n_steps, train_n_ingrs, train_names_d2v50, train_steps_d2v50, train_ingrs_d2v50],
    axis=1,
    ignore_index=True
)

X_test_d2v50 = pd.concat(
    [test_n_steps, test_n_ingrs, test_names_d2v50, test_steps_d2v50, test_ingrs_d2v50],
    axis=1,
    ignore_index=True
)

X_train_d2v100 = pd.concat(
    [train_n_steps, train_n_ingrs, train_names_d2v100, train_steps_d2v100, train_ingrs_d2v100],
    axis=1,
    ignore_index=True
)

X_test_d2v100 = pd.concat(
    [test_n_steps, test_n_ingrs, test_names_d2v100, test_steps_d2v100, test_ingrs_d2v100],
    axis=1,
    ignore_index=True
)

# Check the shape of the training and test sets to ensure everything is in order
print("X_train_cs shape:", X_train_cs.shape)
print("X_test_cs shape:", X_test_cs.shape)
print("X_train_d2v50 shape:", X_train_d2v50.shape)
print("X_test_d2v50 shape:", X_test_d2v50.shape)
print("X_train_d2v100 shape:", X_train_d2v100.shape)
print("X_test_d2v100 shape:", X_test_d2v100.shape)

In [ ]:
# Divide the test sets and train sets
X_trains = [X_train_cs, X_train_d2v50, X_train_d2v100]
y_train = train_labels
X_tests = [X_test_cs, X_test_d2v50, X_test_d2v100]
y_test = test_labels


print(y_train.shape) 
print(y_test.shape)

In [ ]:
# ********Part 2 Train using best Hyperparameter and predict********
# Models to test
# This part need 150 minutes
models = {
    'MLP': MLPClassifier(
        hidden_layer_sizes=(11, ),
        activation='relu',
        solver='sgd',
        alpha=0.01,
        max_iter=10000,
        random_state=123
    ),
    'NB': BernoulliNB(),  
    'DT': DecisionTreeClassifier(max_depth=6, random_state=123),  
    'GBDT': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=123)  
}

# Store performance for comparison
results = {
    'MLP': [],
    'NB': [],
    'DT': [],
    'GBDT': []
}

# Run models and store results
for model_name, model in models.items():
    for X_train, X_test in zip(X_trains, X_tests):
        model.fit(X_train, y_train)
        test_score = model.score(X_test, y_test)  # Use y_test here
        results[model_name].append(test_score)

In [ ]:
# ********Part 3 Results visualization********
# Create a DataFrame for easier plotting
results_df = pd.DataFrame(results, columns=['MLP', 'NB', 'DT', 'GBDT'], index=['CountVec', 'Doc2Vec50', 'Doc2Vec100'])

# Plot the results
plt.figure(figsize=(10, 6))
results_df.plot(kind='bar', rot=0)
plt.title('Model Comparison on Test Sets')
plt.xlabel('Test Set')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.tight_layout()


# Show the plot
plt.show()

In [ ]:
print(results_df)

In [ ]:
results_df.info()